## Import Dependencies

In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)

Loading required package: ggplot2

Loading required package: Rcpp

Loading required package: grf

Loading required package: caret

Loading required package: lattice

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading required package: anytime

Loading required package: rlist

Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: dtw

Loading required package: proxy


Attaching package: ‘proxy’


Th

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

### Read Data

### Transformation on Panel Data

In [2]:
# SET PARAMS
windowsize=2
num_trees = 100



# CREATE OUTPUT FOLDERS
mainDir = "./time_invariant_grf_results"
dir.create(mainDir, showWarnings = FALSE)

subDir = paste("time_invariant_grf_backtest_state_forests_windowsize=",toString(windowsize),"_numtrees=",toString(num_trees),sep="")
outputfolder = file.path(mainDir, subDir)


dir.create(outputfolder, showWarnings = FALSE)

grf.subfolder = paste("time_invariant_grf_grf_windowsize=",toString(windowsize),"_numtrees=",toString(num_trees),sep="")
grf.outputfolder = file.path(mainDir,grf.subfolder)
dir.create(grf.outputfolder, showWarnings = FALSE)

# READ DATA
destfile <- paste("../../data/augmented_us-counties-states_latest_variants",".csv",sep="")
augmented_panel_data <- as.data.frame(fread(file = destfile))
# Time Invariant
hhs_X_w_clusters_fpath = "../benchmark_tcv_kmeans_code/hhs_X_w_clusters.csv"
hhs_X_w_clusters = as.data.frame(fread(file = hhs_X_w_clusters_fpath))

augmented_panel_data <- augmented_panel_data[order(augmented_panel_data$fips, augmented_panel_data$days_from_start), ]
augmented_panel_data <- augmented_panel_data[augmented_panel_data$rolled_cases >= 20,]
augmented_panel_data$log_rolled_cases <- log(augmented_panel_data$rolled_cases + 1.1)
augmented_panel_data <- augmented_panel_data %>%
  group_by(fips) %>%
    mutate(log_rolled_cases_ratios = c(0, diff(log_rolled_cases)))
augmented_panel_data <- augmented_panel_data %>%
  group_by(fips) %>%
    mutate(shifted_days_from_start = days_from_start - first(days_from_start))
augmented_panel_data <- augmented_panel_data[, colSums(!is.na(augmented_panel_data)) > 0]


X_time_invariant <- (hhs_X_w_clusters[,intersect(names(augmented_panel_data), names(hhs_X_w_clusters))])
X_time_invariant <- X_time_invariant[,!names(X_time_invariant) %in% c("county","state")]

start_day = min(augmented_panel_data$days_from_start)
end_day = max(augmented_panel_data$days_from_start)

cutoff_list <- start_day:end_day

### Earlier Days

In [9]:
cutoff=100

check.file.name <- paste0("classical_grf_stateforest_cutoff=", toString(cutoff), ".rds")
check.file.full.name <- file.path(grf.outputfolder, check.file.name)
if (file.exists(check.file.full.name)){
    print(paste0(check.file.name, " exists, skipping"))
    #next
}
print(paste0("Computing Classical GRF for ", toString(cutoff)))
# START TRY

start_time <- Sys.time()
# Given my current cutoff, which block numbers should I use?
fname <- paste("time_invariant_grf_",toString(cutoff),".csv",sep="")        
# SLICE THE DATA ACCORDINGLY
early_data <- augmented_panel_data[augmented_panel_data$days_from_start <= cutoff, ]
case_number_columns <- c("fips","date","county","state","days_from_start","log_rolled_cases", "shifted_days_from_start")
early_data_case_numbers <- early_data[, names(early_data) %in% case_number_columns]
# Format data to be fed to GRF
WYX <- merge(early_data_case_numbers, X_time_invariant, by="fips", all.x=TRUE)
WYX <- WYX[order(WYX$fips, WYX$days_from_start),]

Y <- WYX$log_rolled_cases
W <- WYX$shifted_days_from_start
X <- WYX[, !names(WYX) %in% case_number_columns]

cf <- causal_forest(X,Y,as.numeric(W), num.trees=num_trees, seed = 1)
cf.fname <- check.file.name
cf.path <- file.path(grf.outputfolder, cf.fname)
print(paste0("Saving ", cf.fname))
#saveRDS(cf, cf.path)

# GENERATE r_GRF estimates
WYX_test <- WYX %>%
  group_by(fips) %>%
  filter(days_from_start == max(days_from_start))
X_test <- WYX_test[, !names(WYX_test) %in% case_number_columns]
r_GRF <- predict(cf, X_test)

# Generate predictions
indexing_columns <- c("fips","county","state","date", "days_from_start","shifted_days_from_start", "rolled_cases", "log_rolled_cases")
indexing <- WYX_test[, names(WYX) %in% indexing_columns]
indexing$r_GRF <- unlist(r_GRF)
indexing$predicted_log_rolled_cases_GRF <- indexing$r_GRF*7 + indexing$log_rolled_cases

[1] "classical_grf_stateforest_cutoff=100.rds exists, skipping"
[1] "Computing Classical GRF for 100"
[1] "Saving classical_grf_stateforest_cutoff=100.rds"


In [10]:

X

,LAT,LON,E_UNEMP,E_PCI,E_SNGPNT,E_MUNIT,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,⋯,"Automatic_applications_sent_for_mail-in_ballots_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)",Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),Mention_of_tribal_casinos,Jan_1_2019_Minimum_Wage,Mar_29_2019_Minimum_Wage,Jul_1_2019_Minimum_Wage,Oct_1_2019_Minimum_Wage,Different_Minimum_Wage_for_Smaller_Businesses,positiveScore,hhs_region
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
8,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
1,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
2,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
3,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
7,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
4,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
5,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
6,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4
12,32.53224,-86.64644,1065,29372,1586,886,299,1191,546,15.4,⋯,0,44133,0,7.25,7.25,7.25,7.25,0,0,4


In [11]:
Y

[1]  3.062723  3.082565  3.082565  3.095578  3.114784  3.114784  3.121105
    [8]  3.133629  3.170286  3.205646  3.228543  3.256447  3.283593  3.315224
   [15]  3.350906  3.158215  3.261935  3.360871  3.464396  3.570337  3.694933
   [22]  3.796131  3.902550  4.001341  4.098384  4.195482  4.289872  4.365280
   [29]  4.435398  4.500920  4.554929  4.593242  4.623150  4.649460  4.679084
   [36]  4.710431  4.738327  4.772741  4.810673  4.842724  4.862798  4.868303
   [43]  3.089093  3.188122  3.283593  3.355901  3.409260  3.459915  3.486501
   [50]  3.508128  3.533478  3.062723  3.139833  3.228543  3.299534  3.360871
   [57]  3.390185  3.423332  3.446353  3.473297  3.486501  3.508128  3.520883
   [64]  3.533478  3.075995  3.121105  3.170286  3.211420  3.228543  3.239798
   [71]  3.245379  3.170286  3.335770  3.490864  3.621289  3.256447  3.459915
   [78]  3.628902  3.746811  3.858321  3.958634  4.016898  4.047303  4.067071
   [85]  4.086456  4.105473  4.121821  4.144721  4.182486  4.212551  4.245839
   [92]  4.278054  4.309264  4.322618  4.324511  4.303486  4.264087  4.199777
   [99]  4.119502  3.139833  3.335770  3.578347  3.812045  4.011739  4.162671
  [106]  4.287912  4.409677  4.562412  4.693573  4.803670  4.910236  5.018887
  [113]  5.121154  5.212370  5.274244  5.319310  5.355710  5.383643  5.403128
  [120]  5.415910  5.421609  5.429158  5.432911  5.427903  5.414003  5.394082
  [127]  5.373756  5.355035  5.325585  5.269109  3.095578  3.188122  3.256447
  [134]  3.325550  3.390185  3.459915  3.508128  3.533478  3.541787  3.562263
  [141]  3.570337  3.574350  3.566308  3.558201  3.566308  3.574350  3.578347
  [148]  3.570337  3.562263  3.554123  3.541787  3.049273  3.164269  3.261935
  [155]  3.340841  3.432604  3.075995  3.133629  3.164269  3.193997  3.211420
  [162]  3.228543  3.222868  3.211420  3.199839  3.164269  3.127387  3.095578
  [169]  3.069381  3.062723  3.199839  3.365817  3.529297  3.669769  3.792918
  [176]  3.876248  3.955903  4.039788  4.112512  4.180304  4.251959  4.303486
  [183]  4.357990  4.406196  4.455509  4.497744  4.527517  3.062723  3.095578
  [190]  3.114784  3.133629  3.152125  3.170286  3.182212  3.158215  3.056021
  [197]  3.102021  3.145998  3.199839  3.250928  3.299534  3.340841  3.370738
  [204]  3.399768  3.399768  3.133629  3.222868  3.340841  3.423332  3.495208
  [211]  3.558201  3.594176  3.658788  3.677024  3.669769  3.669769  3.666122
  [218]  3.662462  3.651400  3.617460  3.613617  3.613617  3.613617  3.594176
  [225]  3.582328  3.574350  3.075995  3.121105  3.139833  3.152125  3.152125
  [232]  3.145998  3.158215  3.075995  3.139833  3.176267  3.217160  3.267394
  [239]  3.294248  3.310022  3.340841  3.355901  3.365817  3.390185  3.127387
  [246]  3.228543  3.315224  3.390185  3.459915  3.512398  3.570337  3.640214
  [253]  3.722936  3.799334  3.861331  3.911165  3.964073  4.011739  4.049795
  [260]  4.071953  4.088853  3.062723  3.102021  3.139833  3.193997  3.261935
  [267]  3.345886  3.437208  3.525099  3.601998  3.684226  3.763523  3.840067
  [274]  3.908302  3.966782  4.003951  4.037270  4.067071  4.091245  4.103116
  [281]  4.107825  4.107825  4.114847  3.095578  3.139833  3.158215  3.158215
  [288]  3.350906  3.520883  3.677024  3.821473  3.958634  4.064621  4.149239
  [295]  4.225164  4.299615  4.357990  4.393920  4.418324  4.443827  4.470332
  [302]  4.511958  4.548902  4.587443  4.621746  4.641227  4.649460  4.643979
  [309]  4.621746  4.604742  4.578680  3.089093  3.217160  3.335770  3.441791
  [316]  3.525099  3.640214  3.089093  3.145998  3.211420  3.272823  3.365817
  [323]  3.437208  3.508128  3.554123  3.613617  3.655101  3.691376  3.702007
  [330]  3.722936  3.082565  3.182212  3.272823  3.350906  3.418663  3.490864
  [337]  3.550028  3.598095  3.625103  3.636458  3.640214  3.655101  3.062723
  [344]  3.069381  3.056021  3.049273  3.145998  3.267394  3.385358  3.495208
  [351]  3.590242  3.687807  3.779960  3.852273  3.933784  3.993471  4.039788
  [358]  4.076811  4.10782

In [12]:
indexing

fips,date,county,state,days_from_start,log_rolled_cases,shifted_days_from_start,r_GRF,predicted_log_rolled_cases_GRF
<dbl>,<IDate>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1001,2020-04-30,Autauga,Alabama,100,3.350906,14,0.030717198,3.565926
1003,2020-04-30,Baldwin,Alabama,100,4.868303,26,0.058703279,5.279226
1005,2020-04-30,Barbour,Alabama,100,3.533478,8,0.064040805,3.981764
1007,2020-04-30,Bibb,Alabama,100,3.533478,12,0.042707133,3.832428
1009,2020-04-30,Blount,Alabama,100,3.245379,6,0.037515252,3.507985
1013,2020-04-30,Butler,Alabama,100,3.621289,3,0.046124857,3.944163
1015,2020-04-30,Calhoun,Alabama,100,4.119502,23,0.037887214,4.384712
1017,2020-04-30,Chambers,Alabama,100,5.269109,30,0.063229034,5.711712
1021,2020-04-30,Chilton,Alabama,100,3.541787,20,0.027098959,3.731480


In [13]:
analysis_WYX <- WYX
analysis_WYX$r_tc_predictions <- unlist(cf$predictions)
analysis_WYX

,fips,date,county,state,days_from_start,log_rolled_cases,shifted_days_from_start,LAT,LON,E_UNEMP,⋯,Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),Mention_of_tribal_casinos,Jan_1_2019_Minimum_Wage,Mar_29_2019_Minimum_Wage,Jul_1_2019_Minimum_Wage,Oct_1_2019_Minimum_Wage,Different_Minimum_Wage_for_Smaller_Businesses,positiveScore,hhs_region,r_tc_predictions
,<dbl>,<IDate>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,"<dbl[,1]>"
8,1001,2020-04-16,Autauga,Alabama,86,3.062723,0,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.03220264
1,1001,2020-04-17,Autauga,Alabama,87,3.082565,1,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.03178858
2,1001,2020-04-18,Autauga,Alabama,88,3.082565,2,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.03408014
3,1001,2020-04-19,Autauga,Alabama,89,3.095578,3,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.02975961
7,1001,2020-04-20,Autauga,Alabama,90,3.114784,4,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.02980316
4,1001,2020-04-21,Autauga,Alabama,91,3.114784,5,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.02992165
5,1001,2020-04-22,Autauga,Alabama,92,3.121105,6,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.03021922
6,1001,2020-04-23,Autauga,Alabama,93,3.133629,7,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.02847353
12,1001,2020-04-24,Autauga,Alabama,94,3.170286,8,32.53224,-86.64644,1065,⋯,44133,0,7.25,7.25,7.25,7.25,0,0,4,0.03131413


In [14]:
per_fips_mean_tau <- aggregate(analysis_WYX$r_tc_predictions, list(analysis_WYX$fips), mean)
names(per_fips_mean_tau) <- c("fips", "mean_tau_per_fips")
per_fips_mean_tau

fips,mean_tau_per_fips
<dbl>,<dbl>
1001,0.031001721
1003,0.058257026
1005,0.063661846
1007,0.042525428
1009,0.037770027
1013,0.045424653
1015,0.037793837
1017,0.062984628
1021,0.027616586


In [15]:
diff = as.vector(indexing$r_GRF) - as.vector(per_fips_mean_tau$mean_tau_per_fips)

In [16]:
sum(abs(diff))/length(diff)

[1] 0.0005874995

In [17]:
sum((diff)**2)/length(diff)

[1] 1.101476e-06

In [18]:
max(abs(diff))

[1] 0.01309921

In [19]:
2**2

[1] 4

In [20]:
analysis_WYX[, names(analysis_WYX) %in% c("fips","shifted_days_from_start","r_tc_predictions")]

,fips,shifted_days_from_start,r_tc_predictions
,<dbl>,<dbl>,"<dbl[,1]>"
8,1001,0,0.03220264
1,1001,1,0.03178858
2,1001,2,0.03408014
3,1001,3,0.02975961
7,1001,4,0.02980316
4,1001,5,0.02992165
5,1001,6,0.03021922
6,1001,7,0.02847353
12,1001,8,0.03131413


In [21]:
predict(cf)$predictions - cf$predictions # These gives a tau(x) per fips x time combination
# predict(cf, X_test) # This gives a single tau(x)

0
0
0
0
0
0
0
0
0
0
0


In [22]:
predict(cf, X_test)

predictions
<dbl>
0.030717198
0.058703279
0.064040805
0.042707133
0.037515252
0.046124857
0.037887214
0.063229034
0.027098959


In [ ]:
help(predict)

In [29]:
cf_single <- causal_forest(as.matrix(X$LAT),Y, W, num.trees=100, seed=1)

In [30]:
cf_single$predictions

0.06024304
0.05849363
0.05827931
0.05917859
0.05969659
0.06041000
0.05852497
0.05787173
0.06033763
0.06135448
0.05927999


In [32]:
predict(cf_single, as.matrix(X$LAT))

predictions
<dbl>
0.05985108
0.05985108
0.05985108
0.05985108
0.05985108
0.05985108
0.05985108
0.05985108
0.05985108
